In [ ]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))


[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
!nvidia-smi


Sun Apr 19 06:50:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             11W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import pandas as pd

# Upload the CSV file to Colab first (via Files tab or code below)
from google.colab import files
uploaded = files.upload()
path = list(uploaded.keys())[0]  # Gets the uploaded file name dynamically

df = pd.read_csv(path)
df.head()
df.shape
df.columns
df.isnull().sum()


Saving SpamAssasin.csv to SpamAssasin.csv


,0
sender,0
receiver,210
date,0
subject,16
body,1
label,0
urls,0


In [ ]:
df['text_combined'] = df['subject'].fillna('') + ' ' + df['body'].fillna('')
data = df[['text_combined', 'label']].copy()
data.rename(columns={'text_combined': 'text'}, inplace=True)
data.head()

,text,label
0,"Re: New Sequences Window Date: Wed, 21 ...",0
1,[zzzzteana] RE: Alexander Martin A posted:\nTa...,0
2,[zzzzteana] Moscow bomber Man Threatens Explos...,0
3,[IRR] Klez: The Virus That Won't Die Klez: Th...,0
4,Re: [zzzzteana] Nothing like mama used to make...,0


In [ ]:
import re
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def clean_text(t):
    t = str(t).lower()
    t = re.sub(r'<.*?>', ' ', t)
    t = re.sub(r'[^a-zA-Z]', ' ', t)
    t = ' '.join(w for w in t.split() if w not in stop_words)
    return t

data['clean_text'] = data['text'].apply(clean_text)
data.head()


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


,text,label,clean_text
0,"Re: New Sequences Window Date: Wed, 21 ...",0,new sequences window date wed aug chris garrig...
1,[zzzzteana] RE: Alexander Martin A posted:\nTa...,0,zzzzteana alexander martin posted tassos papad...
2,[zzzzteana] Moscow bomber Man Threatens Explos...,0,zzzzteana moscow bomber man threatens explosio...
3,[IRR] Klez: The Virus That Won't Die Klez: Th...,0,irr klez virus die klez virus die already prol...
4,Re: [zzzzteana] Nothing like mama used to make...,0,zzzzteana nothing like mama used make adding c...


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

X_train, X_test, y_train, y_test = train_test_split(
    data['clean_text'], data['label'], test_size=0.2, random_state=42
)

vectorizer = TfidfVectorizer(max_features=5000)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)


In [ ]:
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

model = LinearSVC()
model.fit(X_train_vec, y_train)

y_pred = model.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


Accuracy: 0.9853700516351118
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       835
           1       0.98      0.96      0.97       327

    accuracy                           0.99      1162
   macro avg       0.99      0.98      0.98      1162
weighted avg       0.99      0.99      0.99      1162



In [ ]:
from transformers import BertTokenizer
import tensorflow as tf


!pip install transformers -q

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize_batch(texts):
    return tokenizer(
        list(texts),
        padding=True,
        truncation=True,
        max_length=128
    )

train_encodings = tokenize_batch(X_train)
test_encodings = tokenize_batch(X_test)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
!pip install transformers torch -q

import torch
from transformers import BertTokenizer, BertForSequenceClassification
import numpy as np
from sklearn.model_selection import train_test_split

# Reuse your existing tokenizer/encodings
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
train_encodings = tokenize_batch(X_train)
test_encodings = tokenize_batch(X_test)

print("PyTorch BERT ready!")


PyTorch BERT ready!


In [ ]:
import torch.nn as nn

class BertLSTMClassifier(nn.Module):
    def __init__(self, bert_model_name='bert-base-uncased', num_classes=1, lstm_hidden=128):
        super().__init__()
        self.bert = BertForSequenceClassification.from_pretrained(bert_model_name, num_labels=num_classes)
        self.lstm = nn.LSTM(768, lstm_hidden, bidirectional=True, batch_first=True)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(lstm_hidden*2, num_classes)
        self.sigmoid = nn.Sigmoid()

    def forward(self, input_ids, attention_mask):
        outputs = self.bert.bert(input_ids, attention_mask=attention_mask)
        lstm_out, (hn, cn) = self.lstm(outputs.last_hidden_state)
        pooled = hn[-2:].transpose(0,1).contiguous().view(hn.size(1), -1)
        dropped = self.dropout(pooled)
        logits = self.classifier(dropped)
        return self.sigmoid(logits)

# Initialize model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = BertLSTMClassifier().to(device)
print(f"Model on: {device}")


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model on: cuda


In [ ]:
from torch.utils.data import DataLoader, TensorDataset

# Convert encodings and labels to tensors
train_input_ids = torch.tensor(train_encodings['input_ids'])
train_attention_mask = torch.tensor(train_encodings['attention_mask'])
train_labels = torch.tensor(y_train.values).long()

test_input_ids = torch.tensor(test_encodings['input_ids'])
test_attention_mask = torch.tensor(test_encodings['attention_mask'])
test_labels = torch.tensor(y_test.values).long()

# Create TensorDatasets
train_dataset = TensorDataset(train_input_ids, train_attention_mask, train_labels)
test_dataset = TensorDataset(test_input_ids, test_attention_mask, test_labels)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)

print(f" Batch=16 Ready: Train={len(train_loader)} batches")
print(f"GPU Free: ~{14 - torch.cuda.memory_reserved()/1e9:.1f}GB")

 Batch=16 Ready: Train=291 batches
GPU Free: ~13.5GB


In [ ]:
from torch.optim import Adam
import torch.nn as nn

optimizer = Adam(model.parameters(), lr=2e-5)
criterion = nn.BCELoss()
device = next(model.parameters()).device

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for batch_idx, batch in enumerate(loader):
        input_ids = batch[0].to(device)
        attention_mask = batch[1].to(device)
        labels = batch[2].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs.squeeze(), labels.float())
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        predicted = (outputs > 0.5).float()
        total += labels.size(0)
        correct += (predicted.squeeze() == labels).sum().item()

        del input_ids, attention_mask, labels, outputs
        torch.cuda.empty_cache()

    return total_loss/len(loader), correct/total

print("TRAINING STARTED (2 epochs, batch=16)...")
for epoch in range(2):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
    print(f"Epoch {epoch+1}/2: Loss={train_loss:.4f}, Acc={train_acc:.4f}")
    torch.cuda.empty_cache()


TRAINING STARTED (2 epochs, batch=16)...
Epoch 1/2: Loss=0.1877, Acc=0.9314
Epoch 2/2: Loss=0.0595, Acc=0.9862


In [ ]:
def evaluate(model, loader, device):
    model.eval()
    correct, total = 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch[0].to(device)
            attention_mask = batch[1].to(device)
            labels = batch[2].to(device)

            outputs = model(input_ids, attention_mask)
            predicted = (outputs > 0.5).float()

            total += labels.size(0)
            correct += (predicted.squeeze() == labels).sum().item()
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            del input_ids, attention_mask, labels, outputs
            torch.cuda.empty_cache()

    from sklearn.metrics import classification_report
    print(f" FINAL Test Accuracy: {correct/total:.4f}")
    print("\n Detailed Report:")
    print(classification_report(all_labels, all_preds))
    return correct/total

test_acc = evaluate(model, test_loader, device)


 FINAL Test Accuracy: 0.9819

 Detailed Report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       835
           1       0.97      0.96      0.97       327

    accuracy                           0.98      1162
   macro avg       0.98      0.98      0.98      1162
weighted avg       0.98      0.98      0.98      1162



In [ ]:
def predict_email(text, model, tokenizer, device):
    model.eval()
    cleaned = clean_text(text)
    encoding = tokenizer(cleaned, return_tensors='pt',
                        padding=True, truncation=True, max_length=128)
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    with torch.no_grad():
        output = model(input_ids, attention_mask)
        prob = output.squeeze().cpu().numpy()

    return "PHISHING" if prob > 0.5 else "LEGITIMATE", prob

# Test it!
test_email = "Click here to claim your $1000 prize! Urgent action required."
result, confidence = predict_email(test_email, model, tokenizer, device)
print(f"Email: {test_email}")
print(f"Prediction: {result} (confidence: {confidence:.3f})")


Email: Click here to claim your $1000 prize! Urgent action required.
Prediction: PHISHING (confidence: 0.992)
